# 🚀 Kaggle GPU Server — Qwen3.8-27B UNCENSORED (HauhauCS Aggressive)

**27B, fully uncensored** (`uncensored` tag on HF), Q4_K_P, split across **2x T4 GPUs** via Ollama.
Speed note: 27B ≈ 20-40 tok/s on 2x T4 — slower than 8B, but far more capable + zero refusals.

**Setup:** right sidebar → **Accelerator: GPU T4 x2** → **Internet: ON** → run cells step by step.


In [ ]:
# 1. Cleanup purani files + GPU check (2x T4 chahiye)
import shutil, subprocess, os, glob
for f in glob.glob("*.gguf"):
    if "Aggressive" not in f:
        print(f"purani file delete: {f} ({os.path.getsize(f)/1e9:.1f} GB free)")
        os.remove(f)
if shutil.which("nvidia-smi") is None:
    raise SystemExit("\n❌ NO GPU! Sidebar \u2192 Accelerator \u2192 'GPU T4 x2', Internet ON.\n")
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True)
print(out.stdout.strip())
n = out.stdout.count("Tesla T4")
print(f"\n✅ {n}x Tesla T4" if n >= 2 else "\n⚠️ 27B ke liye 2x T4 behtar hai!")


In [ ]:
# 2. Install Ollama
!apt-get update -qq && apt-get install -y -qq zstd
!rm -f /usr/local/bin/ollama
!curl -L --retry 3 https://ollama.com/install.sh | sh
!ollama --version


In [ ]:
# 3. Start Ollama server
import shutil, subprocess, time, os, urllib.request
if shutil.which("ollama") is None:
    raise SystemExit("❌ ollama install nahi hua \u2014 cell 2 dobara chalao.")
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
time.sleep(2)
subprocess.Popen(["nohup", "ollama", "serve"],
                 stdout=open("/tmp/ollama.log", "w"),
                 stderr=subprocess.STDOUT, start_new_session=True,
                 env=dict(os.environ))
for i in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/", timeout=3)
        print("✅ Ollama server UP")
        break
    except Exception:
        print(f"waiting... ({i+1})"); time.sleep(2)
else:
    raise SystemExit("❌ server up nahi hua.")


## 4. Download — ~17 GB, live progress bar (sirf pehli dafa per session)


In [ ]:
# 4. Download Qwen3.8-27B Uncensored (HauhauCS Aggressive, Q4_K_P)
import os, subprocess
gguf = "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-Q4_K_P.gguf"
url = "https://huggingface.co/HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF/resolve/main/" + gguf
if os.path.exists(gguf) and os.path.getsize(gguf) > 15_000_000_000:
    print("Model already present, skipping.")
else:
    print("Downloading ~17 GB \u2014 bar dekho, 5-10 min lag sakte hain:")
    r = subprocess.run(["wget", "--show-progress", "-c", "-O", gguf, url])
    size = os.path.getsize(gguf) if os.path.exists(gguf) else 0
    if r.returncode != 0 or size < 15_000_000_000:
        raise SystemExit(f"❌ Download FAILED (size={size/1e9:.1f} GB). Re-run.")
    print(f"\n✅ Downloaded ({size/1e9:.1f} GB)")


In [ ]:
# 5. Ollama mein load karo — 2 GPUs par split hoga
with open("Modelfile", "w") as f:
    f.write("FROM ./Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-Q4_K_P.gguf\n")
    f.write("PARAMETER num_ctx 4096\n")
    f.write("PARAMETER temperature 0.8\n")
!ollama create hauhau-27b -f Modelfile
print("\n--- ollama ps (dono GPU par % dikhna chahiye) ---")
!ollama ps
print("\n--- nvidia-smi (dono mein memory.used > 0) ---")
!nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv


In [ ]:
# 6. Warmup — 27B pehli dafa load hone mein 3-6 min le sakta hai, ghabrana nahi
import subprocess, time
t0 = time.time()
r = subprocess.run(["ollama", "run", "hauhau-27b", "Say 'WARMUP OK' and nothing else."],
                   capture_output=True, text=True, timeout=900)
print(r.stdout.strip())
print(f"\n✅ Warmup {time.time()-t0:.0f}s \u2014 model VRAM mein, ab tez.")


In [ ]:
# 7. ⚡ SPEED TEST
import json, subprocess, time
payload = {"model": "hauhau-27b",
           "messages": [{"role": "user",
                           "content": "Write a Python quicksort with a short explanation."}],
           "max_tokens": 300, "stream": False}
t0 = time.time()
r = subprocess.run(["curl", "-s", "http://127.0.0.1:11434/v1/chat/completions",
                    "-H", "Content-Type: application/json",
                    "-d", json.dumps(payload)],
                   capture_output=True, text=True, timeout=600)
dt = time.time() - t0
data = json.loads(r.stdout)
toks = data.get("usage", {}).get("completion_tokens", 0)
print(data["choices"][0]["message"]["content"][:500])
print(f"\n⚡ SPEED: {toks} tokens / {dt:.1f}s = {toks/dt:.1f} tok/s")
print("(27B on 2x T4: 20-40 tok/s normal hai.)")


In [ ]:
# 8. Cloudflare Tunnel install
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb


## 9. 🌐 Public URL — Forge connect

**Forge → Settings:**
- **API base URL:** `<tunnel-url>/v1`
- **API key:** `ollama`
- **Model:** `hauhau-27b`
→ Save → Test connection → LIVE-OK ✅


In [ ]:
# 9. Tunnel (AAKHRI — chalne do)
import subprocess, time
print("Tunnel start...")
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434",
                      "--http-host-header", "localhost:11434"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(10)
while True:
    line = p.stdout.readline()
    if not line:
        break
    if "trycloudflare.com" in line:
        print("\n🌐 TUMHARA URL:\n" + line.strip() + "\n")
